# Gymnasium: BipedalWalker-v3

Our objective is to train an agent to navigate the BipedalWalker environment using Reinforcement Learning. Before implementing complex algorithms or aiming for advanced maneuvers (like doing a flip), we need to understand the environment's dynamics.

## Environment Overview
`BipedalWalker-v3` is a 2D physics simulation environment from the Gymnasium Box2D environments. The goal is to make a bipedal robot walk to the right end of the terrain. In the `hardcore=True` version, the terrain is not flat; it includes obstacles such as ladders, stumps, and pitfalls.

### Action Space
The action space is a continuous `Box(-1.0, 1.0, (4,), float32)`. The agent controls the robot by applying torques to its four main joints. The four values in the action array represent:
1. Hip 1 (Torque / Speed)
2. Knee 1 (Torque / Speed)
3. Hip 2 (Torque / Speed)
4. Knee 2 (Torque / Speed)

All action values must be within the `[-1.0, 1.0]` range.

### Reward System
The agent receives rewards based on its forward progress and energy efficiency. According to the official documentation, the reward is calculated as follows:
* **Forward Movement:** The agent is rewarded for moving forward (to the right). Reaching the end of the terrain yields a total of over 300 points.
* **Falling Penalty:** If the robot's main body (hull) touches the ground, it falls. This results in a heavy penalty of **-100** points, and the episode terminates immediately.
* **Motor Torque Penalty:** To encourage efficient, natural walking rather than chaotic flailing, applying motor torque costs a small negative reward.
### Set up the Environment

In [1]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import gymnasium as gym
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") # Human we can see the environment

## Baseline: Random Actions

To establish a baseline and visualize how an untrained agent interacts with the physics engine, we will run a single episode using a random policy. The agent will sample actions uniformly from the action space until the episode ends.

An episode ends if:
- `terminated` is True (the agent falls or reaches the goal).
- `truncated` is True (the agent runs out of time/steps).
- Past 20 seconds of simulation time.

In [21]:
import time

env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") 
limit_time = 10  # seconds
start_time = time.time()
obs, info = env.reset()

terminated = False
truncated = False
total_reward = 0.0
step_count = 0

# Loop until the agent finishes or fails
while not (terminated or truncated) and (time.time() - start_time < limit_time):
    # Sample a random continuous action within [-1, 1] for the 4 joints
    action = env.action_space.sample() 
    
    # Step the environment forward
    obs, reward, terminated, truncated, info = env.step(action)
    
    total_reward += reward
    step_count += 1

# Close the rendering window
env.close()

print(f"Episode finished after {step_count} steps.")
print(f"Total Reward with random policy: {total_reward:.2f}")

Episode finished after 52 steps.
Total Reward with random policy: -105.10


### Changing the Environment
Instead of training the agent only to walk, we can reshape the task so it learns to perform a flip.
The main idea is to change the reward and encourage trunk rotation, airtime, and landing control.


Changes in the robot:
- Track the robot body's orientation.
- Reward angular velocity and rotation progress.
- Give a large bonus when the agent completes a full rotation.
- Reduce the fall penalty so the agent is willing to take risks.
- Penalize forward movement less, so the policy focuses on flipping rather than walking.

In [11]:
import gymnasium as gym
import numpy as np

class BipedalFlipperWrapper(gym.Wrapper):
    def __init__(self, env, max_steps=1500):
        super().__init__(env)
        self.max_steps = max_steps
        
        # Expand observation space by 1 to include cumulative_angle
        low = np.append(self.env.observation_space.low, -np.inf)
        high = np.append(self.env.observation_space.high, np.inf)
        self.observation_space = gym.spaces.Box(low, high, dtype=np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.cumulative_angle = 0.0
        self.prev_angle = obs[0]  
        self.flip_completed = False
        self.step_counter = 0
        
        # Append normalized cumulative angle to observation
        obs = np.append(obs, self.cumulative_angle / (2 * np.pi)).astype(np.float32)
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        self.step_counter += 1

        current_angle = obs[0]
        delta_angle = current_angle - self.prev_angle

        # Handle angle wrap-around
        if delta_angle > np.pi:
            delta_angle -= 2 * np.pi
        elif delta_angle < -np.pi:
            delta_angle += 2 * np.pi

        prev_cumulative = self.cumulative_angle
        self.cumulative_angle += delta_angle
        self.prev_angle = current_angle

        custom_reward = 0.0

        # 1. Reward exclusively for frontflip progress
        rotation_progress = self.cumulative_angle - prev_cumulative
        custom_reward += max(0, rotation_progress) * 20.0

        # 2. Penalty for inverting direction (Kills the rocking chair exploit)
        if rotation_progress < -0.05:
            custom_reward -= 1.0

        # 3. Small bonus in the air
        if obs[8] == 0.0 and obs[13] == 0.0:
            custom_reward += 0.02

        # 4. Standard fall penalty if it hasn't completed the flip
        if reward == -100 and not self.flip_completed:
            custom_reward -= 100.0

        # Time limit termination
        if self.step_counter >= self.max_steps:
            truncated = True

        # 5. Flip completion
        # We terminate immediately so it locks in the 500 points and doesn't crash afterward
        if self.cumulative_angle >= 2 * np.pi and not self.flip_completed:
            self.flip_completed = True
            custom_reward += 500.0
            terminated = True

        info['flip_completed'] = self.flip_completed
        info['cumulative_angle'] = self.cumulative_angle

        # Append normalized cumulative angle to observation
        obs = np.append(obs, self.cumulative_angle / (2 * np.pi)).astype(np.float32)

        return obs, custom_reward, terminated, truncated, info

## Integrating Stable Baselines 3

Using Stable Baselines 3, we can implement a Proximal Policy Optimization (PPO) agent to learn how to flip in the BipedalWalker environment.

In [12]:
import os
import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

class DiagnosticCallback(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)

    def _on_step(self) -> bool:
        if "absolute_rotation_progress" in self.locals["infos"][0]:
            # Extract absolute progress to see if it is approaching 6.28 (2*pi)
            abs_progress = [info.get("absolute_rotation_progress", 0) for info in self.locals["infos"]]
            flips = [1 if info.get("flip_completed", False) else 0 for info in self.locals["infos"]]
            
            self.logger.record("custom/mean_abs_rotation", sum(abs_progress)/len(abs_progress))
            self.logger.record("custom/flip_completion_rate", sum(flips)/len(flips))
        return True

def make_env():
    def _init():
        e = gym.make("BipedalWalker-v3", hardcore=False)
        return BipedalFlipperWrapper(e)
    return _init

num_envs = 30
print(f"Creating {num_envs} parallel environments for PPO...")
vec_env = DummyVecEnv([make_env() for _ in range(num_envs)])

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    torch.backends.cudnn.benchmark = True

model_path = "ppo_bipedal_flipper.zip"
if os.path.exists(model_path):
    print(f"Loading existing model from {model_path}...")
    model = PPO.load(model_path, env=vec_env, device=device)
else:
    print("Initializing new PPO model on device", device)
    # PPO hyperparameters tuned for parallel continuous control
    model = PPO("MlpPolicy", vec_env, verbose=1, device=device, 
                n_steps=2048, batch_size=256, ent_coef=0.01, learning_rate=3e-4,
                tensorboard_log="./ppo_bipedal_tensorboard/")

# Increased training budget to 10M steps
total_steps = 1_000_000
print(f"Starting training for {total_steps} timesteps...")

callback = DiagnosticCallback()

model.learn(total_timesteps=total_steps, callback=callback)
print("Training finished! Saving model...")
model.save("ppo_bipedal_flipper")
vec_env.close()

Creating 30 parallel environments for PPO...
Using device: cpu
Initializing new PPO model on device cpu
Using cpu device
Starting training for 1000000 timesteps...
Logging to ./ppo_bipedal_tensorboard/PPO_4
------------------------------
| time/              |       |
|    fps             | 3689  |
|    iterations      | 1     |
|    time_elapsed    | 16    |
|    total_timesteps | 61440 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 2964        |
|    iterations           | 2           |
|    time_elapsed         | 41          |
|    total_timesteps      | 122880      |
| train/                  |             |
|    approx_kl            | 0.004802545 |
|    clip_fraction        | 0.0368      |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.74       |
|    explained_variance   | -0.00597    |
|    learning_rate        | 0.0003      |
|    loss                 | 30

### Testing Model


In [14]:
import time
import os
import gymnasium as gym
import torch
from stable_baselines3 import PPO

model_path = "ppo_bipedal_flipper.zip"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Create a single human-render environment and wrap it
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human")
env = BipedalFlipperWrapper(env)

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found: {model_path}. Train and save the model before running tests.")

print(f"Loading model from {model_path}...")
model = PPO.load(model_path, device=device)

num_episodes = 5
max_seconds = 15

for ep in range(1, num_episodes + 1):
    obs, info = env.reset()
    start_time = time.time()
    done = False
    total_reward = 0.0
    step_count = 0

    print(f"\n=== Test Episode {ep} ===")
    while not done and (time.time() - start_time) < max_seconds:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
        step_count += 1

    flip_completed = info.get("flip_completed", False)
    final_angle = info.get("cumulative_angle", 0.0)

    print(f"Episode {ep} — steps: {step_count}, total_reward: {total_reward:.2f}")
    print(f"Flip completed: {flip_completed}, Final cumulative rotation: {final_angle:.2f} radians")

    time.sleep(0.5)

env.close()
print("All tests finished.")

Using device: cpu
Loading model from ppo_bipedal_flipper.zip...

=== Test Episode 1 ===
Episode 1 — steps: 729, total_reward: 222.02
Flip completed: False, Final cumulative rotation: 1.25 radians

=== Test Episode 2 ===
Episode 2 — steps: 729, total_reward: 216.26
Flip completed: False, Final cumulative rotation: 0.79 radians

=== Test Episode 3 ===
Episode 3 — steps: 730, total_reward: 216.50
Flip completed: False, Final cumulative rotation: 0.62 radians

=== Test Episode 4 ===
Episode 4 — steps: 731, total_reward: 222.64
Flip completed: False, Final cumulative rotation: 1.17 radians

=== Test Episode 5 ===
Episode 5 — steps: 729, total_reward: 229.79
Flip completed: False, Final cumulative rotation: 1.06 radians
All tests finished.
